# Notebook 00 — Configuración del Entorno en SEPAL.io y Auditoría de Assets GEE

**Tesis:** *Evaluación de un modelo geoespacial de imágenes Sentinel-1 y Sentinel-2 para la estimación de cobertura manglar* (Maestría en Ciencias en Geomática — USAC).

- **Instancia SEPAL recomendada:** `t2` (2 CPU, 4 GB RAM, 0.04 USD/h) o `m2` (0.10 USD/h).
- **Propósito:**
  1. Verificar recursos de la máquina virtual de SEPAL (`/home` persistente vs. `/tmp` temporal) e instalar dependencias faltantes.
  2. Autenticar e inicializar **Google Earth Engine (GEE)** en Python.
  3. Comprobar que los **Assets de AOI** (`AOI_PACIFICO`, `AOI_CARIBE`) y la **capa guía de manglar actual** (`projects/tesismangle/assets/DinMangle_2020_2022_V23`) son accesibles y se superponen correctamente.

In [ ]:
# 1. Instalar dependencias faltantes (ejecutar solo la primera vez en SEPAL)
# Descomenta la siguiente linea si falta algun paquete en tu entorno de SEPAL:
# !pip install -q -r ../requirements.txt

import os
import platform
import shutil
import sys
from pathlib import Path

# Asegurar que la raiz del repositorio este en sys.path
ROOT = Path.cwd().resolve()
if not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Python: {sys.version.split()[0]} | SO: {platform.system()} {platform.release()}")
print(f"CPU disponibles: {os.cpu_count()}")
print(f"Raiz del proyecto: {ROOT}")

for ruta in [Path.home(), Path("/tmp")]:
    if ruta.exists():
        total, used, free = shutil.disk_usage(ruta)
        print(f"Disco {str(ruta):15s} -> Libre: {free/1e9:6.1f} GB / Total: {total/1e9:6.1f} GB")

In [ ]:
# 2. Verificar librerias clave del pipeline
paquetes = [
    "ee", "geemap", "ipywidgets", "yaml", "numpy", "pandas",
    "scipy", "sklearn", "statsmodels", "matplotlib", "seaborn", "geopandas", "openpyxl",
]
faltantes = []
for pkg in paquetes:
    try:
        mod = __import__(pkg)
        ver = getattr(mod, "__version__", "OK")
        print(f"  [OK] {pkg:14s} ({ver})")
    except ImportError:
        faltantes.append(pkg)
        print(f"  [FALTA] {pkg}")

if faltantes:
    print("\nEjecuta en la celda anterior: !pip install -q -r ../requirements.txt")

In [ ]:
# 3. Cargar config.yaml, crear estructura de salidas e inicializar GEE
import ee
import pandas as pd
from src.config_utils import (
    check_asset_exists,
    ensure_output_dirs,
    init_gee,
    load_config,
)

cfg = load_config()
dirs = ensure_output_dirs(cfg)
init_gee(cfg)

print("\nProyecto GEE activo:", cfg["gee"]["project"])
print("Raiz de assets de salida:", cfg["gee"]["asset_root"])
print("Capa guia de mangle:", cfg["assets"]["mangle_guia"])
print("AOIs configurados:", cfg["assets"]["aoi"])

In [ ]:
# 4. Auditar existencia y estructura de los Assets clave en GEE
assets_a_verificar = {
    "AOI_PACIFICO": cfg["assets"]["aoi"]["PACIFICO"],
    "AOI_CARIBE": cfg["assets"]["aoi"]["CARIBE"],
    "MANGLE_GUIA": cfg["assets"]["mangle_guia"],
}

diagnosticos = []
for etiqueta, asset_id in assets_a_verificar.items():
    info = check_asset_exists(asset_id)
    info["etiqueta"] = etiqueta
    diagnosticos.append(info)

df_assets = pd.DataFrame(diagnosticos)
display(df_assets[["etiqueta", "asset_id", "existe", "tipo"] + [c for c in ["bandas", "n_features", "propiedades", "error"] if c in df_assets.columns]])

if not df_assets["existe"].all():
    print(
        "\n[ATENCION] Alguno de los assets no se encontro con la ruta actual en config.yaml.\n"
        "Edita 'assets.aoi' o 'gee.project' en config.yaml con los Asset IDs exactos y vuelve a correr esta celda."
    )

In [ ]:
# 5. Inspeccion visual rapida de los AOIs y la capa guia DinMangle_2020_2022_V23
import geemap

m = geemap.Map(center=[14.7, -90.3], zoom=7)
m.add_basemap("SATELLITE")

# Agregar AOIs disponibles
for reg, aoi_id in cfg["assets"]["aoi"].items():
    info = check_asset_exists(aoi_id)
    if info["existe"]:
        fc = ee.FeatureCollection(aoi_id) if info["tipo"] == "TABLE" else ee.Image(aoi_id).geometry()
        m.addLayer(fc, {"color": "00FFFF"}, f"AOI {reg}")

# Agregar capa guia de mangle (soporta tanto IMAGE como TABLE)
guia_id = cfg["assets"]["mangle_guia"]
info_guia = check_asset_exists(guia_id)
if info_guia["existe"]:
    if info_guia["tipo"] == "IMAGE":
        img_guia = ee.Image(guia_id)
        banda0 = img_guia.bandNames().get(0)
        print("Bandas de DinMangle_2020_2022_V23:", info_guia["bandas"])
        m.addLayer(img_guia.select([banda0]).selfMask(), {"min": 1, "max": 5, "palette": ["00FF66", "FFFF00", "FF3333", "00AAFF", "AA00FF"]}, "Guia DinMangle 2020-2022")
    else:
        fc_guia = ee.FeatureCollection(guia_id)
        m.addLayer(fc_guia, {"color": "00FF66"}, "Guia DinMangle 2020-2022")

m